In [10]:
!pip install -q pandas numpy scikit-learn joblib

In [11]:
# Import os for handling file paths and operating-system-related tasks
import os

# Import NumPy for generating random numerical data and performing calculations
import numpy as np

# Import Pandas for creating and manipulating the faculty workload dataset
import pandas as pd

# Import MinMaxScaler to normalize workload features into a common range from 0 to 1
from sklearn.preprocessing import MinMaxScaler

In [12]:
def generate_faculty_dataset(
    num_records=500,  # Number of faculty records to generate
    output_path="/content/faculty_workload_dataset.csv",  # Location to save the dataset
    random_state=42  # Seed value for reproducible results
):
    # Set the random seed so the same dataset can be generated again
    np.random.seed(random_state)

    # Generate unique IDs for each faculty member
    faculty_ids = [
        f"FAC_{i:03d}"
        for i in range(1, num_records + 1)
    ]

    # Generate synthetic faculty workload values
    teaching_hours = np.random.randint(8, 36, num_records)
    advising_students = np.random.randint(2, 31, num_records)
    committee_count = np.random.randint(0, 8, num_records)
    research_hours = np.random.randint(2, 21, num_records)
    admin_hours = np.random.randint(1, 16, num_records)

    # Generate semester progress between 0.05 and 1.00
    semester_progress = np.round(
        np.random.uniform(0.05, 1.00, num_records), 2
    )

    # Generate historical leave days between 0 and 15
    historical_leave_days = np.random.randint(0, 16, num_records)

    # Create a DataFrame using the generated faculty data
    df = pd.DataFrame({
        "Faculty_ID": faculty_ids,
        "Teaching_Hours": teaching_hours,
        "Advising_Students": advising_students,
        "Committee_Count": committee_count,
        "Research_Hours": research_hours,
        "Admin_Hours": admin_hours,
        "Semester_Progress": semester_progress,
        "Historical_Leave_Days": historical_leave_days
    })

    # --------------------------------------------------------
    # Normalize workload-related features
    # --------------------------------------------------------

    # Create a MinMaxScaler to convert values into the range 0 to 1
    scaler = MinMaxScaler()

    # Select the numerical workload columns for normalization
    score_columns = [
        "Teaching_Hours",
        "Advising_Students",
        "Committee_Count",
        "Research_Hours",
        "Admin_Hours"
    ]

    # Fit the scaler and transform the selected workload features
    normalized_values = scaler.fit_transform(df[score_columns])

    # Create a new DataFrame for the normalized scores
    normalized_df = pd.DataFrame(
        normalized_values,
        columns=[
            "Teaching_Score",
            "Advising_Score",
            "Committee_Score",
            "Research_Score",
            "Admin_Score"
        ]
    )

    # Combine the original dataset and normalized score columns
    df = pd.concat(
        [df.reset_index(drop=True), normalized_df],
        axis=1
    )

    # --------------------------------------------------------
    # Calculate workload score
    # --------------------------------------------------------

    # Calculate the overall workload score using weighted features
    df["Workload_Score"] = (
        0.30 * df["Teaching_Score"] +
        0.20 * df["Advising_Score"] +
        0.15 * df["Committee_Score"] +
        0.20 * df["Research_Score"] +
        0.15 * df["Admin_Score"]
    )

    # --------------------------------------------------------
    # Calculate pressure score
    # --------------------------------------------------------

    # Calculate pressure using workload, semester progress,
    # historical leave days, and teaching score
    df["Pressure_Score"] = (
        0.45 * df["Workload_Score"] +
        0.25 * df["Semester_Progress"] +
        0.20 * (df["Historical_Leave_Days"] / 15) +
        0.10 * df["Teaching_Score"]
    )

    # --------------------------------------------------------
    # Create burnout-risk labels
    # --------------------------------------------------------

    # Convert the numerical pressure score into burnout categories
    def assign_burnout_risk(score):
        # Scores below 0.40 are classified as Low risk
        if score < 0.40:
            return "Low"

        # Scores from 0.40 to below 0.65 are classified as Medium risk
        elif score < 0.65:
            return "Medium"

        # Scores of 0.65 or above are classified as High risk
        else:
            return "High"

    # Apply the burnout-risk function to every pressure score
    df["Burnout_Risk"] = df["Pressure_Score"].apply(
        assign_burnout_risk
    )

    # --------------------------------------------------------
    # Select final columns before saving
    # --------------------------------------------------------

    # Define the columns that should be included in the final dataset
    final_columns = [
        "Faculty_ID",
        "Teaching_Hours",
        "Advising_Students",
        "Committee_Count",
        "Research_Hours",
        "Admin_Hours",
        "Semester_Progress",
        "Historical_Leave_Days",
        "Workload_Score",
        "Pressure_Score",
        "Burnout_Risk"
    ]

    # Keep only the required final columns
    df = df[final_columns]

    # --------------------------------------------------------
    # Save the dataset
    # --------------------------------------------------------

    # Save the DataFrame as a CSV file
    df.to_csv(output_path, index=False)

    # Display the save location and dataset dimensions
    print(f"Dataset saved successfully at: {output_path}")
    print(f"Dataset shape: {df.shape}")

    # Return the generated dataset
    return df

In [13]:
# Call the dataset-generation function to create the synthetic faculty workload dataset
# The generated dataset is stored in the DataFrame variable named df
df = generate_faculty_dataset()

Dataset saved successfully at: /content/faculty_workload_dataset.csv
Dataset shape: (500, 11)


In [14]:
# Display the first 10 rows of the faculty workload dataset
# This helps verify that the dataset was generated correctly
df.head(10)

,Faculty_ID,Teaching_Hours,Advising_Students,Committee_Count,Research_Hours,Admin_Hours,Semester_Progress,Historical_Leave_Days,Workload_Score,Pressure_Score,Burnout_Risk
0,FAC_001,14,23,7,10,4,0.07,8,0.487698,0.365853,Low
1,FAC_002,27,12,6,4,14,0.24,4,0.572619,0.441382,Medium
2,FAC_003,22,29,3,12,15,0.06,3,0.673810,0.410066,Medium
3,FAC_004,18,6,5,7,9,0.67,1,0.388095,0.392513,Low
4,FAC_005,15,2,6,3,14,0.90,0,0.356746,0.411462,Medium
5,FAC_006,28,9,1,8,14,0.28,10,0.499603,0.502229,Medium
6,FAC_007,14,22,2,19,7,0.93,6,0.505556,0.562222,Medium
7,FAC_008,33,29,0,3,1,0.11,5,0.481746,0.403545,Medium
8,FAC_009,26,13,4,18,14,0.94,14,0.681349,0.794940,High
9,FAC_010,30,13,4,12,12,0.38,14,0.637698,0.650112,High


In [15]:
# Display a summary of the dataset
# This shows the column names, number of non-null values, and data types
# It also helps identify missing values and understand the dataset structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Faculty_ID             500 non-null    object 
 1   Teaching_Hours         500 non-null    int64  
 2   Advising_Students      500 non-null    int64  
 3   Committee_Count        500 non-null    int64  
 4   Research_Hours         500 non-null    int64  
 5   Admin_Hours            500 non-null    int64  
 6   Semester_Progress      500 non-null    float64
 7   Historical_Leave_Days  500 non-null    int64  
 8   Workload_Score         500 non-null    float64
 9   Pressure_Score         500 non-null    float64
 10  Burnout_Risk           500 non-null    object 
dtypes: float64(3), int64(6), object(2)
memory usage: 43.1+ KB


In [16]:
# Count the number of missing values in each column
# This helps check whether the dataset contains any null or missing data
df.isnull().sum()

,0
Faculty_ID,0
Teaching_Hours,0
Advising_Students,0
Committee_Count,0
Research_Hours,0
Admin_Hours,0
Semester_Progress,0
Historical_Leave_Days,0
Workload_Score,0
Pressure_Score,0


In [17]:
# Count the number of records in each burnout-risk category
# This shows how many faculty members are classified as Low, Medium, or High risk
df["Burnout_Risk"].value_counts()

,count
Burnout_Risk,
Medium,314
Low,112
High,74


In [18]:
# Import the files module from Google Colab
# This module is used to download files from the Colab environment
from google.colab import files

# Download the generated faculty workload dataset to the local computer
files.download("/content/faculty_workload_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>